# 06 - Cash Flow Lending LGD: PD-LGD Alignment

This notebook demonstrates the **PD-aligned Cash Flow Lending LGD engine**, which integrates the PD Scorecard (WoE logistic regression, score bands A-E) with LGD estimation for 8 unsecured/receivables-secured cash flow lending products.

**Key features:**
- PD score band segmentation (A-E) aligned with WoE scorecard
- PD-band-adjusted downturn scalars and margins of conservatism
- DSCR stress overlay and conduct classification overlay
- Industry risk integration (16 industries)
- Product-specific supervisory LGD floors
- PD-LGD internal consistency validation (APRA APS 113)

**APRA compliance:** PD and LGD must be internally consistent -- higher PD borrowers should face systematically higher LGD due to weaker cash flow capacity during workout.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

from src.data_generation import (
    generate_cashflow_lending_data, CASHFLOW_PRODUCTS,
    PD_SCORE_BANDS, CONDUCT_CLASSES, INDUSTRIES, INDUSTRY_RISK_PROFILES,
)
from src.lgd_calculation import CashFlowLendingLGDEngine
from src.validation import (
    generate_validation_report, calibration_by_score_band,
    pd_lgd_consistency_check, accuracy_metrics, discriminatory_power,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

loans, cashflows = generate_cashflow_lending_data(n_loans=400, seed=45)
engine = CashFlowLendingLGDEngine()
loans_overlay = engine.apply_overlays(loans)

print(f"Cash flow lending portfolio: {len(loans)} loans, {len(cashflows)} cashflows")
print(f"Products: {loans['cashflow_product'].nunique()}")
print(f"Industries: {loans['industry'].nunique()}")
print(f"PD score bands: {sorted(loans['pd_score_band'].unique())}")

## 1. Portfolio Overview and PD Score Band Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PD score band distribution
band_counts = loans["pd_score_band"].value_counts().reindex(["A","B","C","D","E"])
axes[0,0].bar(band_counts.index, band_counts.values, color=["#2ecc71","#27ae60","#f39c12","#e74c3c","#c0392b"])
axes[0,0].set_title("PD Score Band Distribution")
axes[0,0].set_ylabel("Count")

# Product distribution
prod_counts = loans["cashflow_product"].value_counts()
axes[0,1].barh(prod_counts.index, prod_counts.values, color="#3498db")
axes[0,1].set_title("Cash Flow Lending Products")
axes[0,1].set_xlabel("Count")

# PD distribution
axes[1,0].hist(loans["pd_estimate"], bins=30, color="#9b59b6", edgecolor="white", alpha=0.8)
axes[1,0].set_title("PD Estimate Distribution")
axes[1,0].set_xlabel("PD")
axes[1,0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Realised LGD by PD band
band_order = ["A","B","C","D","E"]
bp_data = [loans[loans.pd_score_band == b]["realised_lgd"].values for b in band_order]
bp = axes[1,1].boxplot(bp_data, labels=band_order, patch_artist=True)
colors = ["#2ecc71","#27ae60","#f39c12","#e74c3c","#c0392b"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1,1].set_title("Realised LGD by PD Score Band")
axes[1,1].set_ylabel("Realised LGD")
axes[1,1].set_xlabel("PD Score Band")

plt.tight_layout()
plt.show()

## 2. LGD Overlay Pipeline Waterfall

The CashFlowLendingLGDEngine applies overlays in sequence:
1. Industry recovery haircut
2. PD-band-adjusted downturn scalar
3. PD-band-adjusted MoC + industry adjustment
4. DSCR stress overlay
5. Conduct overlay
6. Supervisory LGD floor

In [ ]:
# Waterfall: mean LGD at each pipeline step
steps = {
    "Realised LGD": loans_overlay["realised_lgd"].mean(),
    "+ Industry Haircut": loans_overlay["lgd_industry_adjusted"].mean(),
    "x Downturn Scalar": loans_overlay["lgd_downturn"].mean(),
    "+ MoC": (loans_overlay["lgd_downturn"] + loans_overlay["industry_moc"]).mean(),
    "+ DSCR Stress": (loans_overlay["lgd_downturn"] + loans_overlay["industry_moc"] + loans_overlay["dscr_stress_overlay"]).mean(),
    "+ Conduct": (loans_overlay["lgd_downturn"] + loans_overlay["industry_moc"] + loans_overlay["dscr_stress_overlay"] + loans_overlay["conduct_overlay"]).mean(),
    "Final (floored)": loans_overlay["lgd_final"].mean(),
}

fig, ax = plt.subplots(figsize=(12, 5))
x = list(steps.keys())
y = list(steps.values())
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c", "#9b59b6", "#f39c12", "#1abc9c"]
bars = ax.bar(x, y, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, y):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Mean LGD")
ax.set_title("LGD Overlay Pipeline Waterfall (Portfolio Average)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 3. PD-LGD Alignment: LGD by PD Score Band

The core PD-LGD alignment principle: LGD should increase monotonically across PD bands A through E, reflecting the fact that higher-PD borrowers have weaker cash flow capacity during workout.

In [ ]:
# PD-LGD calibration by score band
cal = calibration_by_score_band(loans_overlay, band_col="pd_score_band")
print("PD Score Band Calibration:")
print(cal["calibration_table"].to_string(index=False))
print(f"\nMonotonic LGD across bands: {cal['is_monotonic']}")
print(f"PD-LGD Spearman correlation: {cal.get('pd_lgd_correlation', 'N/A')}")

if "el_table" in cal:
    print("\nImplied Expected Loss by Band:")
    print(cal["el_table"][["pd_score_band","mean_pd","mean_lgd","implied_el","count"]].to_string(index=False))

In [ ]:
# Visualise PD-LGD alignment
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Mean LGD by PD band (realised vs final)
band_order = ["A", "B", "C", "D", "E"]
realised_by_band = loans_overlay.groupby("pd_score_band")["realised_lgd"].mean().reindex(band_order)
final_by_band = loans_overlay.groupby("pd_score_band")["lgd_final"].mean().reindex(band_order)

x = np.arange(len(band_order))
w = 0.35
axes[0].bar(x - w/2, realised_by_band, w, label="Realised LGD", color="#3498db", alpha=0.8)
axes[0].bar(x + w/2, final_by_band, w, label="Final LGD (with overlays)", color="#e74c3c", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(band_order)
axes[0].set_title("Mean LGD by PD Score Band")
axes[0].set_ylabel("LGD")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

# Panel 2: Implied EL by band
if "el_table" in cal:
    el = cal["el_table"]
    axes[1].bar(el["pd_score_band"].astype(str), el["implied_el"], color="#9b59b6")
    axes[1].set_title("Implied Expected Loss by Band")
    axes[1].set_ylabel("EL = PD x LGD")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Panel 3: PD vs LGD scatter
sc = axes[2].scatter(loans_overlay["pd_estimate"], loans_overlay["lgd_final"],
                     c=loans_overlay["industry_risk_score"], cmap="RdYlGn_r",
                     alpha=0.5, s=20, edgecolors="none")
axes[2].set_xlabel("PD Estimate")
axes[2].set_ylabel("Final LGD")
axes[2].set_title("PD vs LGD (colour = industry risk)")
axes[2].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.colorbar(sc, ax=axes[2], label="Industry Risk Score")

plt.tight_layout()
plt.show()

## 4. PD-LGD Internal Consistency Check (APRA APS 113)

In [ ]:
consistency = pd_lgd_consistency_check(loans_overlay)

print("=== PD-LGD Internal Consistency Check ===\n")
corr = consistency["correlation"]
print(f"Spearman rho:  {corr['spearman_rho']:.4f}  (p={corr['p_value']:.6f})")
print(f"Positive:      {corr['is_positive']}")
print(f"Interpretation: {corr['interpretation']}")
print(f"\nPortfolio Implied EL: {consistency['portfolio_implied_el']:.4%}")
print(f"EL Monotonic across bands: {consistency['el_monotonic']}")
print("\nEL by PD Score Band:")
print(consistency["el_by_band"][["pd_score_band","n","mean_pd","mean_lgd","implied_el"]].to_string(index=False))

## 5. Product-Level LGD Analysis

In [ ]:
# LGD by product and PD band heatmap
pivot = loans_overlay.pivot_table(
    values="lgd_final", index="cashflow_product",
    columns="pd_score_band", aggfunc="mean"
).reindex(columns=["A","B","C","D","E"])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("PD Score Band")
ax.set_title("Mean Final LGD by Product x PD Band")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                    color="white" if val > 0.7 else "black", fontsize=9)

plt.colorbar(im, ax=ax, label="Mean LGD", format=mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

# Product supervisory floors
print("\nProduct Supervisory LGD Floors:")
for prod, floor in sorted(engine.PRODUCT_LGD_FLOOR.items(), key=lambda x: x[1]):
    print(f"  {prod:30s} {floor:.0%}")

## 6. DSCR Stress and Conduct Overlays

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DSCR stress overlay
dscr_range = np.linspace(0.5, 3.0, 100)
stress = np.maximum((1.3 - dscr_range) * 0.02, 0)
axes[0].plot(dscr_range, stress * 100, color="#e74c3c", linewidth=2)
axes[0].fill_between(dscr_range, 0, stress * 100, alpha=0.2, color="#e74c3c")
axes[0].axvline(1.3, color="gray", linestyle="--", alpha=0.5, label="DSCR = 1.3x threshold")
axes[0].set_xlabel("DSCR")
axes[0].set_ylabel("Stress Overlay (pp)")
axes[0].set_title("DSCR Stress Overlay: max(0, (1.3 - DSCR) x 2%)")
axes[0].legend()

# Conduct overlay impact
conduct_lgd = loans_overlay.groupby("conduct_classification")["lgd_final"].mean()
conduct_lgd = conduct_lgd.reindex(["Green", "Amber", "Red"])
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
axes[1].bar(conduct_lgd.index, conduct_lgd.values, color=colors)
axes[1].set_ylabel("Mean Final LGD")
axes[1].set_title("Mean LGD by Conduct Classification")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
for i, (idx, val) in enumerate(conduct_lgd.items()):
    overlay = engine.CONDUCT_OVERLAY[idx]
    axes[1].text(i, val + 0.005, f"+{overlay:.1%} overlay", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Cross-Product Comparison: Cash Flow vs Secured Commercial

In [ ]:
from src.data_generation import generate_commercial_data
from src.lgd_calculation import CommercialLGDEngine

com_loans, _ = generate_commercial_data()
com_engine = CommercialLGDEngine()
com_overlay = com_engine.apply_overlays(com_loans)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LGD distributions
axes[0].hist(com_overlay["lgd_final"], bins=30, alpha=0.6, color="#3498db",
             label=f"Secured Commercial (mean={com_overlay['lgd_final'].mean():.1%})", density=True)
axes[0].hist(loans_overlay["lgd_final"], bins=30, alpha=0.6, color="#e74c3c",
             label=f"Cash Flow Lending (mean={loans_overlay['lgd_final'].mean():.1%})", density=True)
axes[0].set_xlabel("Final LGD")
axes[0].set_title("LGD Distribution: Secured vs Cash Flow Lending")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

# Industry comparison
common_industries = set(com_overlay["industry"].unique()) & set(loans_overlay["industry"].unique())
com_ind = com_overlay.groupby("industry")["lgd_final"].mean()
cfl_ind = loans_overlay.groupby("industry")["lgd_final"].mean()
merged = pd.DataFrame({"Secured Commercial": com_ind, "Cash Flow Lending": cfl_ind}).dropna()
merged = merged.sort_values("Cash Flow Lending", ascending=True)

y = np.arange(len(merged))
h = 0.35
axes[1].barh(y - h/2, merged["Secured Commercial"], h, label="Secured Commercial", color="#3498db")
axes[1].barh(y + h/2, merged["Cash Flow Lending"], h, label="Cash Flow Lending", color="#e74c3c")
axes[1].set_yticks(y)
axes[1].set_yticklabels(merged.index, fontsize=8)
axes[1].set_xlabel("Mean Final LGD")
axes[1].set_title("LGD by Industry: Secured vs Cash Flow")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 8. Full Validation Report

In [ ]:
report = generate_validation_report(
    loans_overlay, segment_col="pd_score_band"
)

print("=== Cash Flow Lending LGD Validation Report ===\n")

print("Accuracy Metrics:")
for k, v in report["accuracy"].items():
    print(f"  {k:20s}: {v}")

print("\nDiscriminatory Power:")
for k, v in report["discriminatory_power"].items():
    print(f"  {k:20s}: {v}")

print("\nConservatism Test:")
for k, v in report["conservatism"].items():
    print(f"  {k:20s}: {v}")

if "stability_psi" in report:
    print(f"\nPSI: {report['stability_psi']['PSI']:.6f} ({report['stability_psi']['Interpretation']})")

if "pd_lgd_consistency" in report:
    c = report["pd_lgd_consistency"]
    print(f"\nPD-LGD Consistency:")
    print(f"  Correlation positive: {c['correlation']['is_positive']}")
    print(f"  EL monotonic:         {c['el_monotonic']}")
    print(f"  Portfolio EL:         {c['portfolio_implied_el']:.4%}")

## 9. APRA Compliance Checklist

| Requirement | Status |
|---|---|
| PD-LGD internal consistency (APS 113) | Positive Spearman correlation, monotonic EL |
| Downturn LGD (APS 113) | PD-band-adjusted scalars by product |
| Margin of conservatism | PD-band-scaled MoC (0.85x-1.25x base) |
| Supervisory LGD floors (APS 112) | Product-specific floors (35%-55%) |
| Industry risk differentiation | 16-industry risk scores integrated |
| Cash flow capacity drivers | DSCR stress overlay, conduct overlay |
| EL = PD x LGD monotonicity | Verified across score bands A-E |